# Decision Tree


* 여러분은 OO 통신화사 데이터분석가 입니다.
* 회사는 약정기간이 끝난 고객이 번호이동(이탈)해 가는 문제를 해결하고자 합니다.
* 그래서 여러분에게, 어떤 고객이 번호이동(이탈)해 가는지 예측 모델링을 의뢰하였습니다.

![](https://d18lkz4dllo6v2.cloudfront.net/cumulus_uploads/entry/23964/mobile%20phones.png)

## 1.환경준비

### (1) import

In [ ]:
#라이브러리들을 불러오자.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

### (2) 데이터 준비

In [ ]:
# 데이터를 불러옵시다.
path = 'https://raw.githubusercontent.com/DA4BAM/dataset/master/mobile_cust_churn.csv'
data = pd.read_csv(path)
data.head()

* 변수설명
    * COLLEGE : 대학 졸업여부
    * INCOME : 연수입
    * OVERAGE : 월평균 초과사용 시간(분)
    * LEFTOVER : 월평균 잔여시간비율(%)
    * HOUSE : 집값
    * HANDSET_PRICE : 스마트폰 가격
    * OVER_15MINS_CALLS_PER_MONTH : 월평균 장기통화(15분이상) 횟수
    * AVERAGE_CALL_DURATION : 평균 통화 시간
    * REPORTED_SATISFACTION : 만족도 설문조사 결과
    * REPORTED_USAGE_LEVEL : 사용도 자가진단 결과
    * CONSIDERING_CHANGE_OF_PLAN : 향후 변경계획 설문조사 결과
    * CHURN : 이탈(번호이동) 여부 (Target 변수)


## 2.데이터 전처리

### (1) 변수정리
* 불필요한 변수를 정리합시다.
    * 식별자 : 일련번호, 주민번호, 전화번호, 고객ID, 사번
    * 시계열 데이터 중 : 어떤 기간동안 거의 변화가 없는 값.(변동이 거의 없는) 값.


In [ ]:
drop_cols = ['id']
data.drop(drop_cols, axis = 1, inplace = True )

### (2) x,y 분할

In [ ]:
target = 'CHURN'
x = data.drop(target, axis = 1)
y = data.loc[:, target]

### (3) 가변수화

가변수화를 수행하시오.

In [ ]:
dumm_cols = ['REPORTED_SATISFACTION','REPORTED_USAGE_LEVEL','CONSIDERING_CHANGE_OF_PLAN']
x = pd.get_dummies(x, columns = dumm_cols, drop_first = True)

### (4) train, val 분할

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=.2, random_state = 20)

## 3.모델링

### (1) 필요한 함수 불러오기

In [ ]:
# 모델링을 위해
from sklearn.tree import DecisionTreeClassifier

# 평가를 위해.
from sklearn.metrics import *

### (2) 선언

In [ ]:
model = DecisionTreeClassifier()

### (3) 모델링(학습)

In [ ]:
model.fit(x_train, y_train)

### (4) 검증 : 예측

In [ ]:
pred = model.predict(x_val)

In [ ]:
pred

### (5) 검증 : 평가

* confusion matrix

In [ ]:
confusion_matrix(y_val, pred)

* classification report

In [ ]:
print(classification_report(y_val, pred, digits = 4))

## 4.Decision Tree 추가 내용

### (1) 모델 시각화

In [ ]:
x_train.columns

In [ ]:
# 시각화
from sklearn.tree import plot_tree

# Decision Tree는 모델을 시각화 할 수 있습니다.
# 약 2분 소요됨
plot_tree(model,                       # 만든 모델 이름
          feature_names = x_train.columns,    #Feature 이름, list(x_train)
          filled = True);

* 모델을 작게 만들어 봅시다.

In [ ]:
model2 = DecisionTreeClassifier(max_depth = 3)
model2.fit(x_train, y_train)

plt.figure(figsize = (20,6)) # 그림 사이즈 조절
plot_tree(model2, feature_names = x_train.columns,
          filled = True, fontsize = 10)

### (2) 변수 중요도

In [ ]:
# 변수 중요도
print(x_train.columns)
print(model.feature_importances_)

* 변수중요도 그래프 그리기 함수 만들기

In [ ]:
def plot_feature_importance(importance, names):
    feature_importance = np.array(importance)
    feature_names = np.array(names)

    data={'feature_names':feature_names,'feature_importance':feature_importance}
    fi_df = pd.DataFrame(data)

    fi_df.sort_values(by=['feature_importance'], ascending=False,inplace=True)
    fi_df.reset_index(drop=True, inplace = True)

    plt.figure(figsize=(10,8))
    sns.barplot(x='feature_importance', y='feature_names', data = fi_df)

    plt.xlabel('FEATURE IMPORTANCE')
    plt.ylabel('FEATURE NAMES')
    plt.grid()

In [ ]:
list(x_train)

In [ ]:
plot_feature_importance(model.feature_importances_, list(x_train))

### (3) 실습 Hyper Parameter 다루기
* 다음의 조건으로 모델 생성, 시각화, 성능 비교를 수행해 봅시다.
    * max_depth : 1,2,3,4,5
